# Solution: consume_02 — manual offsets & groups

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['strom'])

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} offset={msg.offset()} -- processing...', end='')
    consumer.commit(message=msg)
    print(' committed.')
consumer.close()

## Task A — without commit, the same messages reappear on restart

In [ ]:
# Same as above but *without* consumer.commit(...) — run twice with the same group.id.
from confluent_kafka import Consumer
consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':           'no-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['strom'])

for i in range(3):
    msg = consumer.poll(2.0)
    if msg and not msg.error():
        print(f'#{i+1} offset={msg.offset()} -- (NOT committing)')
consumer.close()
# Re-run this cell — the same offsets show up again.

## Task B — two consumers, same group

In [ ]:
# Run this cell in two tabs simultaneously, then produce events in a third tab.
from confluent_kafka import Consumer
from datetime import datetime
consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',
})
consumer.subscribe(['strom', 'wasser'])
try:
    while True:
        msg = consumer.poll(0.5)
        if msg and not msg.error():
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] {msg.topic()} P{msg.partition()} offset={msg.offset()}')
except KeyboardInterrupt:
    consumer.close()